# Обучение предобученной модели FinBert

In [44]:
#transformers==4.32.0
#accelerate==0.27.2
#torch=='2.12.0+cu130' (!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130)
#datasets==5.0.0
#huggingface_hub==0.36.2
#multiprocess

In [ ]:
import datasets
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report, accuracy_score
import torch

In [3]:
torch.__version__, transformers.__version__

('2.12.0+cu130', '4.32.0')

In [4]:
torch.cuda.is_available()

True

### Преобразование данных

In [5]:
import re

def show_label_statistics(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    label_count = {}
    for line in lines:
        line = line.strip()
        if not line:
            continue
        label = line.split()[1]
        label_count[label] = label_count.get(label, 0) + 1
    
    print("Статистика меток:")
    for label, count in sorted(label_count.items()):
        print(f"   {label}: {count}")

def convert_to_bio(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    converted = []
    previous_entity_type = None
    
    for line in lines:
        line = line.strip()
        if not line:
            converted.append('')
            previous_entity_type = None
            continue
        
        parts = line.split(' - ')
        if len(parts) != 2:
            continue
        
        token_part = parts[0].strip() 
        label = parts[1].strip()  
        
        token = token_part.split()[0]
        
        if label != 'O':
            entity_type = label.split('-')[1] 
            
            if previous_entity_type != entity_type:
                label = f'B-{entity_type}'
            
            previous_entity_type = entity_type
        else:
            previous_entity_type = None
        
        converted.append(f"{token} {label}")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(converted))
    
    print(f"Конвертация завершена! Файл сохранен: {output_file}")
    
    show_label_statistics(output_file)


convert_to_bio('data/train.txt', 'data/train_mod.txt')

Конвертация завершена! Файл сохранен: data/train_mod.txt
Статистика меток:
   B-LOC: 171
   B-MISC: 7
   B-ORG: 243
   B-PER: 747
   I-LOC: 185
   I-ORG: 141
   I-PER: 36
   O: 39480


In [6]:
convert_to_bio('data/test.txt', 'data/test_mod.txt')

Конвертация завершена! Файл сохранен: data/test_mod.txt
Статистика меток:
   B-LOC: 39
   B-MISC: 7
   B-ORG: 56
   B-PER: 216
   I-LOC: 40
   I-ORG: 58
   I-PER: 20
   O: 12810


In [7]:
from transformers import BertTokenizer, BertForTokenClassification, TrainingArguments, Trainer

labels = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC']
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-pretrain")

model = BertForTokenClassification.from_pretrained(
    "yiyanghkust/finbert-pretrain",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

C:\Users\vacil\kurs\finbert_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at yiyanghkust/finbert-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Сборка датасета

In [8]:
# создание валидаионной выборки

def load_bio_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    data = []
    tokens = []
    ner_tags = []
    
    for line in lines:
        line = line.strip()
        if not line: 
            if tokens:
                data.append({'tokens': tokens, 'ner_tags': ner_tags})
                tokens = []
                ner_tags = []
            continue
        
        parts = line.split()
        if len(parts) != 2:
            print(f"Пропущена строка с неверным форматом: {line}")
            continue
        
        token, tag = parts
        tokens.append(token)
        ner_tags.append(label2id[tag])

    if tokens:
        data.append({'tokens': tokens, 'ner_tags': ner_tags})
    
    return data

def split_train_val_from_file(train_file_path, val_file_path, val_size=200, random_seed=42):
    import random
    
    all_data = load_bio_data(train_file_path)
    
    random.seed(random_seed)
    random.shuffle(all_data)
    
    # отделяем validation
    val_data = all_data[:val_size]
    train_data = all_data[val_size:]
    
    # сохраняем обратно в файлы
    save_bio_data(train_data, train_file_path)
    save_bio_data(val_data, val_file_path)  
    
    print(f"Файлы сохранены:")
    print(f"   {train_file_path}: {len(train_data)} предложений")
    print(f"   {val_file_path}: {len(val_data)} предложений")
    
    return train_data, val_data

def save_bio_data(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data:
            tokens = item['tokens']
            tags = item['ner_tags']
            
            for token, tag_id in zip(tokens, tags):
                tag = id2label[tag_id]
                f.write(f"{token} {tag}\n")
            f.write("\n") 

split_train_val_from_file('data/train_mod.txt', 'data/val_mod.txt', val_size=200)

Файлы сохранены:
   data/train_mod.txt: 964 предложений
   data/val_mod.txt: 200 предложений


([{'tokens': ['Section', '10', '.', '1', 'Notices', '.'],
   'ner_tags': [0, 0, 0, 0, 0, 0]},
  {'tokens': ['Borrower',
    'grants',
    'Agent',
    'for',
    'the',
    'benefit',
    'of',
    'Lenders',
    'a',
    'license',
    'to',
    'enter',
    'and',
    'occupy',
    'any',
    'of',
    'its',
    'premises',
    ',',
    'without',
    'charge',
    ',',
    'to',
    'exercise',
    'any',
    'of',
    'Agent',
    "'",
    's',
    'rights',
    'or',
    'remedies',
    ';',
    '(',
    'e',
    ')',
    'apply',
    'to',
    'the',
    'Obligations',
    'then',
    'due',
    'and',
    'payable',
    'any',
    '(',
    'i',
    ')',
    'balances',
    'and',
    'deposits',
    'of',
    'Borrower',
    'it',
    'holds',
    ',',
    'or',
    '(',
    'ii',
    ')',
    'any',
    'amount',
    'held',
    'by',
    'Agent',
    'or',
    'Lenders',
    'owing',
    'to',
    'or',
    'for',
    'the',
    'credit',
    'or',
    'the',
    'account',
 

In [9]:
train_data = load_bio_data('data/train_mod.txt')
val_data = load_bio_data('data/val_mod.txt')
test_data = load_bio_data('data/test_mod.txt')

print(f"✅ Загружено:")
print(f"   Train: {len(train_data)} предложений")
print(f"   Validation: {len(val_data)} предложений")
print(f"   Test: {len(test_data)} предложений")

✅ Загружено:
   Train: 964 предложений
   Validation: 200 предложений
   Test: 303 предложений


In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding=True, 
        is_split_into_words=True,
        max_length=512
    )
    
    labels_batch = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels_batch.append(label_ids)
    
    tokenized_inputs["labels"] = labels_batch
    return tokenized_inputs

In [11]:
from transformers import AutoTokenizer

from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [12]:
from datasets import Dataset, DatasetDict

In [13]:
datasets = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data)
})

# применение токенизации ко всему датасету
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/964 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

### Обучение FinBERT

In [14]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from seqeval.scheme import IOB2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [15]:
training_args = TrainingArguments(
    output_dir="./finbert-ner-model", 
    learning_rate=2e-5,  
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    evaluation_strategy="epoch", 
    save_strategy="epoch",
    weight_decay=0.01,
    warmup_ratio=0.1, 
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True, 
    metric_for_best_model="eval_f1",
    fp16=True,
)

In [18]:
!pip uninstall accelerate -y

Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


In [19]:
!pip cache purge

Files removed: 12 (1936.6 MB)
Directories removed: 0


In [20]:
!pip install accelerate==0.27.2


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()

C:\Users\vacil\kurs\finbert_env\lib\site-packages\accelerate\accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.110001,1.000000,0.538860,0.700337
2,0.613700,0.053199,0.993590,0.803109,0.888252
3,0.613700,0.035681,0.948276,0.854922,0.899183
4,0.067300,0.034605,0.855721,0.891192,0.873096
5,0.035800,0.026276,0.914894,0.891192,0.902887
6,0.035800,0.028903,0.909574,0.886010,0.897638
7,0.016300,0.026413,0.906250,0.901554,0.903896
8,0.016300,0.027503,0.910526,0.896373,0.903394
9,0.010900,0.028340,0.919786,0.891192,0.905263
10,0.006300,0.027590,0.920213,0.896373,0.908136


TrainOutput(global_step=610, training_loss=0.12320291678436467, metrics={'train_runtime': 255.2132, 'train_samples_per_second': 37.772, 'train_steps_per_second': 2.39, 'total_flos': 1810558112778240.0, 'train_loss': 0.12320291678436467, 'epoch': 10.0})

In [28]:
model.save_pretrained("./finbert-ner-model-saved")
tokenizer.save_pretrained("./finbert-ner-model-saved")
print("Модель сохранена в './finbert-ner-model-saved'")

Модель сохранена в './finbert-ner-model-saved'


### Использование и оценка

In [31]:
from transformers import pipeline
# пример использования
ner_pipeline = pipeline("token-classification", model="./finbert-ner-model-saved", aggregation_strategy="simple")
result = ner_pipeline("Apple Inc. reported record profits of $100 billion.")
print(result)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'ORG', 'score': np.float32(0.6819954), 'word': 'apple', 'start': 0, 'end': 5}]


In [33]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
import torch
import numpy as np
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("./finbert-ner-model-saved", use_fast=True)
model_path = "./finbert-ner-model-saved"


model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval()

labels = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC']
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

print("Модель и токенизатор загружены")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Устройство: {device}")


def load_bio_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    data = []
    tokens = []
    ner_tags = []
    
    for line in lines:
        line = line.strip()
        if not line:
            if tokens:
                data.append({'tokens': tokens, 'ner_tags': ner_tags})
                tokens = []
                ner_tags = []
            continue
        
        parts = line.split()
        if len(parts) != 2:
            continue
        
        token, tag = parts
        tokens.append(token)
        ner_tags.append(label2id[tag])
    
    if tokens:
        data.append({'tokens': tokens, 'ner_tags': ner_tags})
    
    return data

test_data = load_bio_data('data/test_mod.txt')
print(f"Загружено тестовых предложений: {len(test_data)}")


def predict_sentence(tokens, model, tokenizer, device):
    encoding = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True,
        return_tensors="pt",
        max_length=512
    )
    
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)
    
    encoding_with_words = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    word_ids = encoding_with_words.word_ids()
    
    previous_word_idx = None
    pred_tags = []
    
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            pred_tag_idx = predictions[0][idx].item()
            pred_tags.append(id2label[pred_tag_idx])
            previous_word_idx = word_idx
    

    if len(pred_tags) != len(tokens):
        pred_tags = []
        previous_word_idx = None
        for idx, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx != previous_word_idx:
                if idx < predictions.shape[1]:
                    pred_tag_idx = predictions[0][idx].item()
                    pred_tags.append(id2label[pred_tag_idx])
                    previous_word_idx = word_idx
    
    return pred_tags


true_labels = []
pred_labels = []

print("Оценка модели на тестовой выборке...")
for item in tqdm(test_data):
    tokens = item['tokens']
    true_tags = [id2label[tag] for tag in item['ner_tags']]
    
    try:
        pred_tags = predict_sentence(tokens, model, tokenizer, device)
        

        if len(pred_tags) != len(true_tags):

            min_len = min(len(pred_tags), len(true_tags))
            pred_tags = pred_tags[:min_len]
            true_tags = true_tags[:min_len]
        
        if len(pred_tags) > 0 and len(true_tags) > 0:
            true_labels.append(true_tags)
            pred_labels.append(pred_tags)
            
    except Exception as e:
        print(f"Ошибка при обработке: {e}")
        continue

print(f"\nУспешно обработано примеров: {len(true_labels)}")

if len(true_labels) == 0:
    print("Нет данных для оценки")
else:

    print("Результаты на тестовой выборке")

    

    precision = precision_score(true_labels, pred_labels)
    recall = recall_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    

    print("Отчет по каждому классу")

    print(classification_report(true_labels, pred_labels))
    
    print("Примеры предсказаний (первые 5 предложений):")
    
    for i in range(min(5, len(true_labels))):
        tokens = test_data[i]['tokens'][:15]  # Первые 15 токенов
        true = true_labels[i][:15]
        pred = pred_labels[i][:15]
        
        print(f"\nПример {i+1}:")
        print(f"Токены:    {tokens}")
        print(f"Истина:    {true}")
        print(f"Предсказание: {pred}")
        
        errors = [(j, t, p) for j, (t, p) in enumerate(zip(true, pred)) if t != p]
        if errors:
            print(f"Ошибки на позициях: {[(j, t, p) for j, t, p in errors[:5]]}")

Модель и токенизатор загружены
Устройство: cuda
Загружено тестовых предложений: 303
Оценка модели на тестовой выборке...


100%|██████████| 303/303 [00:03<00:00, 90.05it/s]



Успешно обработано примеров: 303
Результаты на тестовой выборке
Precision: 0.6855
Recall:    0.8095
F1-Score:  0.7424
Отчет по каждому классу
              precision    recall  f1-score   support

         LOC       0.37      0.59      0.46        39
        MISC       0.00      0.00      0.00         7
         ORG       0.42      0.54      0.47        56
         PER       0.85      0.95      0.90       213

   micro avg       0.69      0.81      0.74       315
   macro avg       0.41      0.52      0.45       315
weighted avg       0.69      0.81      0.75       315

Примеры предсказаний (первые 5 предложений):

Пример 1:
Токены:    ['Subordinated', 'Loan', 'Agreement', '-', 'Silicium', 'de', 'Provence', 'SAS', 'and', 'Evergreen', 'Solar', 'Inc', '.', '7', '-']
Истина:    ['O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O']
Предсказание: ['O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-ORG', 'I-ORG', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O']

#### Использование на пользовательских примерах

In [34]:
from transformers import pipeline

ner_pipeline = pipeline("token-classification", model="./finbert-ner-model-saved", aggregation_strategy="simple")
result = ner_pipeline("Apple Inc. reported record profits of $100 billion.")
print(result)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'ORG', 'score': np.float32(0.6819954), 'word': 'apple', 'start': 0, 'end': 5}]


In [35]:
ner_pipeline("While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk’s rocket company.")

[{'entity_group': 'ORG',
  'score': np.float32(0.7985156),
  'word': 'spacex',
  'start': 6,
  'end': 12},
 {'entity_group': 'ORG',
  'score': np.float32(0.62588143),
  'word': 'nasa',
  'start': 197,
  'end': 201},
 {'entity_group': 'ORG',
  'score': np.float32(0.57736),
  'word': 'elon musk',
  'start': 262,
  'end': 271}]

In [36]:
ner_pipeline("Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)")

[{'entity_group': 'PER',
  'score': np.float32(0.53502744),
  'word': 'ron bar',
  'start': 36,
  'end': 43},
 {'entity_group': 'LOC',
  'score': np.float32(0.56338954),
  'word': '##on',
  'start': 43,
  'end': 45},
 {'entity_group': 'ORG',
  'score': np.float32(0.66303706),
  'word': 'spacex',
  'start': 69,
  'end': 75},
 {'entity_group': 'ORG',
  'score': np.float32(0.27947134),
  'word': 'ronb',
  'start': 145,
  'end': 149}]

In [37]:
ner_pipeline("John Smith works at Google in New York")

[{'entity_group': 'PER',
  'score': np.float32(0.58316857),
  'word': 'john smith',
  'start': 0,
  'end': 10},
 {'entity_group': 'LOC',
  'score': np.float32(0.35629168),
  'word': 'google',
  'start': 20,
  'end': 26},
 {'entity_group': 'LOC',
  'score': np.float32(0.959636),
  'word': 'new york',
  'start': 30,
  'end': 38}]

In [38]:
ner_pipeline("The borrower Sarah Johnson holds deposits of $50000")

[{'entity_group': 'PER',
  'score': np.float32(0.9964078),
  'word': 'borrower',
  'start': 4,
  'end': 12},
 {'entity_group': 'PER',
  'score': np.float32(0.3115776),
  'word': 'sarah',
  'start': 13,
  'end': 18},
 {'entity_group': 'LOC',
  'score': np.float32(0.47500944),
  'word': 'johnson',
  'start': 19,
  'end': 26}]

In [39]:
ner_pipeline("Microsoft CEO Satya Nadella visited London yesterday")

[{'entity_group': 'ORG',
  'score': np.float32(0.5890674),
  'word': 'microsoft',
  'start': 0,
  'end': 9},
 {'entity_group': 'LOC',
  'score': np.float32(0.3957027),
  'word': 'sat',
  'start': 14,
  'end': 17},
 {'entity_group': 'PER',
  'score': np.float32(0.37204164),
  'word': '##ya',
  'start': 17,
  'end': 19},
 {'entity_group': 'LOC',
  'score': np.float32(0.6422545),
  'word': 'nadella',
  'start': 20,
  'end': 27},
 {'entity_group': 'LOC',
  'score': np.float32(0.92724526),
  'word': 'london',
  'start': 36,
  'end': 42}]

In [40]:
ner_pipeline("This LOAN AND SECURITY AGREEMENT dated January 27, 1999, between SILICON VALLEY BANK ( Bank ), a California - chartered bank with its principal place of business at 3003 Tasman Drive, Santa Clara, California 95054 with a loan production office located at 40 William St., Ste.")

[{'entity_group': 'ORG',
  'score': np.float32(0.99170715),
  'word': 'silicon valley bank',
  'start': 65,
  'end': 84},
 {'entity_group': 'ORG',
  'score': np.float32(0.99453557),
  'word': 'bank',
  'start': 87,
  'end': 91},
 {'entity_group': 'LOC',
  'score': np.float32(0.991655),
  'word': 'california',
  'start': 97,
  'end': 107},
 {'entity_group': 'ORG',
  'score': np.float32(0.99451184),
  'word': 'bank',
  'start': 120,
  'end': 124},
 {'entity_group': 'LOC',
  'score': np.float32(0.9609429),
  'word': '3003 tasman drive',
  'start': 165,
  'end': 182},
 {'entity_group': 'LOC',
  'score': np.float32(0.9949161),
  'word': 'santa clara',
  'start': 184,
  'end': 195},
 {'entity_group': 'LOC',
  'score': np.float32(0.9908024),
  'word': 'california',
  'start': 197,
  'end': 207},
 {'entity_group': 'LOC',
  'score': np.float32(0.99078274),
  'word': '40 william st',
  'start': 255,
  'end': 268}]